# Stage 0 — Test the trained tokenizer

Verifies `./vi_smollm_tokenizer/` (Byte-Level BPE, 32k vocab) against the checks in `docs/stage0_tokenizer.md`.

**Prerequisite:** tokenizer trained — `uv run python data/tokenizer_train.py`

Paths below are relative to the **repo root** — the first cell locates it
(works whether Jupyter was launched from the repo root or from `notebooks/`).

In [11]:
import os
import sys

def _find_repo_root():
    d = os.getcwd()
    while not os.path.isdir(os.path.join(d, "finetuning")):
        parent = os.path.dirname(d)
        if parent == d:
            raise RuntimeError("Cannot find the ViRL-SmolLM2-135M repo root")
        d = parent
    return d

REPO_ROOT = _find_repo_root()
sys.path.insert(0, REPO_ROOT)
os.chdir(REPO_ROOT)

In [12]:
from transformers import PreTrainedTokenizerFast

tokenizer = PreTrainedTokenizerFast.from_pretrained("./vi_smollm_tokenizer")
print("✅ Loaded tokenizer from ./vi_smollm_tokenizer")

✅ Loaded tokenizer from ./vi_smollm_tokenizer


In [13]:
# 1. Vocab size
assert tokenizer.vocab_size == 32000, f"Expected 32000, got {tokenizer.vocab_size}"
print(f"✅ Vocab size: {tokenizer.vocab_size}")

✅ Vocab size: 32000


In [14]:
# 2. Special tokens
required = ["<unk>", "<s>", "</s>", "<pad>", "<mask_token>", "<|im_start|>", "<|im_end|>"]
vocab = tokenizer.get_vocab()
missing = [tok for tok in required if tok not in vocab]
assert not missing, f"Missing special tokens: {missing}"
print("✅ Special tokens OK")
for tok in required:
    print(f"  {tok!r:16} -> id {vocab[tok]}")

✅ Special tokens OK
  '<unk>'          -> id 0
  '<s>'            -> id 1
  '</s>'           -> id 2
  '<pad>'          -> id 3
  '<mask_token>'   -> id 4
  '<|im_start|>'   -> id 5
  '<|im_end|>'     -> id 6


In [15]:
# 3. Encode / decode Vietnamese with diacritics
text = "Xin chào, đây là thử nghiệm tiếng Việt có dấu: ì ạch"
ids = tokenizer.encode(text)
decoded = tokenizer.decode(ids)
assert text in decoded or decoded in text, "Encode/decode mismatch"
print(f"✅ Encode/decode OK: '{decoded}'")
print(f"   {len(ids)} tokens: {ids}")

✅ Encode/decode OK: 'Xin chào, đây là thử nghiệm tiếng Việt có dấu: ì ạch'
   14 tokens: [4975, 2607, 18, 650, 288, 1638, 1010, 905, 631, 303, 1684, 32, 16867, 26245]


In [16]:
# 4. Chat template
assert tokenizer.chat_template is not None, "Missing chat template"
print("✅ Chat template OK")

messages = [
    {"role": "system", "content": "You are a helpful Vietnamese assistant."},
    {"role": "user", "content": "Xin chào!"},
]
rendered = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
assert "<|im_start|>" in rendered and "<|im_end|>" in rendered
print(rendered)

✅ Chat template OK
<|im_start|>system
You are a helpful Vietnamese assistant.<|im_end|>
<|im_start|>user
Xin chào!<|im_end|>
<|im_start|>assistant



In [17]:
# 5. Output files
expected = {"tokenizer.json", "tokenizer_config.json"}
files = set(os.listdir("./vi_smollm_tokenizer"))
missing = expected - files
assert not missing, f"Missing files: {missing}"
print("✅ Output files OK")
for f in sorted(files):
    print(f"  {f}")

✅ Output files OK
  chat_template.jinja
  tokenizer.json
  tokenizer_config.json
